# Qwen Kaggle API — Abliterated GGUF

Strata-free experiment: build upstream `llama.cpp`, load a Qwen3.8-Flash-Next abliterated GGUF, then expose the local OpenAI-compatible API through an authenticated Cloudflare Quick Tunnel.

**Important:** this is an initial bootstrap notebook, not yet validated end-to-end. Start with a Kaggle GPU session. Attach a Kaggle Dataset containing the GGUF shards if possible; the model is too large to casually duplicate into `/kaggle/working`. The API and tunnel only start after the local server passes its health check.

In [ ]:
# ===== CONFIGURATION =====
MODEL_ROOT = "/kaggle/input"  # attach a Kaggle Dataset containing the GGUF shards
MODEL_FILE = ""  # optional: absolute path to the first shard, e.g. ...-00001-of-00002.gguf
LLAMA_CPP_DIR = "/kaggle/working/llama.cpp"
LLAMA_CPP_REF = "master"
CONTEXT_SIZE = 2048
GPU_LAYERS = 999
SERVER_HOST = "127.0.0.1"
SERVER_PORT = 8080
START_CLOUDFLARE_TUNNEL = True

import time, subprocess, secrets, re
from pathlib import Path


## 1. Select the model
Attach the Kaggle Dataset containing the GGUF shards, then run this cell.

In [ ]:
gguf_files = sorted(Path(MODEL_ROOT).rglob("*.gguf"))
if MODEL_FILE:
    model_path = Path(MODEL_FILE)
    if not model_path.is_file():
        raise FileNotFoundError("MODEL_FILE does not exist: " + str(model_path))
else:
    first_shards = [p for p in gguf_files if "-00001-of-" in p.name.lower()]
    candidates = first_shards or gguf_files
    if len(candidates) != 1:
        raise RuntimeError("Could not select one model. Set MODEL_FILE to the first shard of the desired GGUF.")
    model_path = candidates[0]

print("Model:", model_path)


## 2. Build upstream llama.cpp with CUDA
This is the part that replaces Strata. The build itself uses `/kaggle/working`; it does not create a Strata expert pack.

In [ ]:
llama_dir = Path(LLAMA_CPP_DIR)
if not (llama_dir / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", LLAMA_CPP_REF, "https://github.com/ggml-org/llama.cpp.git", str(llama_dir)], check=True)
else:
    subprocess.run(["git", "-C", str(llama_dir), "fetch", "--depth", "1", "origin", LLAMA_CPP_REF], check=True)
    subprocess.run(["git", "-C", str(llama_dir), "checkout", "FETCH_HEAD"], check=True)
LLAMA_CPP_COMMIT = subprocess.run(["git", "-C", str(llama_dir), "rev-parse", "HEAD"], check=True, capture_output=True, text=True).stdout.strip()
print("llama.cpp commit:", LLAMA_CPP_COMMIT)

# Kaggle may expose the CUDA driver only through the toolkit stubs directory.
# Add existing stub-library locations to CMake search paths so CUDA::cuda_driver can be created.
cuda_roots = [Path("/usr/local/cuda"), Path("/usr/local/cuda-12.8")]
cuda_stub_dirs = []
for root in cuda_roots:
    for relative in ("targets/x86_64-linux/lib/stubs", "lib64/stubs"):
        candidate = root / relative
        if (candidate / "libcuda.so").exists():
            cuda_stub_dirs.append(str(candidate))
cmake_args = [
    "cmake", "-S", str(llama_dir), "-B", str(llama_dir / "build"),
    "-DGGML_CUDA=ON", "-DLLAMA_BUILD_SERVER=ON",
    "-DCMAKE_BUILD_TYPE=Release", "-DCUDAToolkit_ROOT=/usr/local/cuda",
]
if cuda_stub_dirs:
    cmake_args.append("-DCMAKE_LIBRARY_PATH=" + ";".join(cuda_stub_dirs))
    print("CUDA driver stub search path:", ", ".join(cuda_stub_dirs))

# Clear the failed configure cache so a previous missing-library result is not reused.
build_dir = llama_dir / "build"
if (build_dir / "CMakeCache.txt").exists():
    (build_dir / "CMakeCache.txt").unlink()
subprocess.run(cmake_args, check=True)
subprocess.run(["cmake", "--build", str(build_dir), "--config", "Release", "-j", "2", "--target", "llama-server"], check=True)
server_binary = build_dir / "bin" / "llama-server"
if not server_binary.is_file():
    raise FileNotFoundError(f"Build completed but llama-server was not found at {server_binary}")
print("Built:", server_binary)

## 3. Start the local authenticated API
Start with a small context. If model loading fails from RAM/VRAM pressure or unsupported GGUF tensors, stop here and inspect the logs before adding a tunnel.

In [ ]:
import requests
API_KEY = secrets.token_urlsafe(32)
server_log_path = Path("/kaggle/working/llama-server.log")
server_log = server_log_path.open("w")
server_cmd = [str(server_binary), "-m", str(model_path), "--host", SERVER_HOST, "--port", str(SERVER_PORT), "-c", str(CONTEXT_SIZE), "-ngl", str(GPU_LAYERS), "--api-key", API_KEY]
print("Starting llama-server; model loading can take several minutes.")
server_process = subprocess.Popen(server_cmd, stdout=server_log, stderr=subprocess.STDOUT, start_new_session=True)
healthy = False
for _ in range(180):
    if server_process.poll() is not None:
        break
    try:
        response = requests.get(f"http://{SERVER_HOST}:{SERVER_PORT}/health", timeout=3)
        if response.ok:
            healthy = True
            break
    except requests.RequestException:
        pass
    time.sleep(5)
if not healthy:
    server_log.flush()
    print(server_log_path.read_text(errors="replace")[-12000:])
    raise RuntimeError("llama-server did not become healthy. Cloudflare tunnel was not started.")
print("Local health check passed.")
print("Server log:", server_log_path)
print("API key created (kept out of logs). Store it securely before using the endpoint.")

## 4. Test chat completions locally
The API key is sent in the request header and is not printed.

In [ ]:
test_response = requests.post(
    f"http://{SERVER_HOST}:{SERVER_PORT}/v1/chat/completions",
    headers={"Authorization": f"Bearer {API_KEY}"},
    json={"model": "local-model", "messages": [{"role": "user", "content": "Reply with exactly: API test passed"}], "max_tokens": 32},
    timeout=300,
)
test_response.raise_for_status()
test_json = test_response.json()
print(test_json.get("choices", [{}])[0].get("message", {}).get("content", "No message content returned"))

## 5. Optionally expose the API through Cloudflare Quick Tunnel
Only run this after the local health check and chat-completion test pass. Quick Tunnel URLs are temporary and change when the tunnel restarts.

In [ ]:
if START_CLOUDFLARE_TUNNEL:
    import platform, urllib.request
    arch = platform.machine().lower()
    if arch not in ("x86_64", "amd64"):
        raise RuntimeError(f"Unsupported architecture for this cloudflared download cell: {arch}")
    cloudflared = Path("/kaggle/working/cloudflared")
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", cloudflared)
    cloudflared.chmod(0o755)
    tunnel_log_path = Path("/kaggle/working/cloudflared.log")
    tunnel_log = tunnel_log_path.open("w")
    tunnel_process = subprocess.Popen([str(cloudflared), "tunnel", "--no-autoupdate", "--url", f"http://{SERVER_HOST}:{SERVER_PORT}"], stdout=tunnel_log, stderr=subprocess.STDOUT, start_new_session=True)
    public_url = None
    for _ in range(60):
        time.sleep(2)
        tunnel_log.flush()
        tunnel_text = tunnel_log_path.read_text(errors="replace")
        match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", tunnel_text)
        if match:
            public_url = match.group(0)
            break
        if tunnel_process.poll() is not None:
            break
    if not public_url:
        print(tunnel_log_path.read_text(errors="replace")[-8000:])
        raise RuntimeError("Cloudflare Quick Tunnel did not provide a public URL.")
    print("Public API base URL:", public_url)
    print("Use this API key from your client; do not paste it into public logs or commit it to GitHub.")
else:
    print("Tunnel disabled.")

## Client configuration notes
- OpenAI-compatible base URL: append `/v1` to the public tunnel URL.
- API key: the value held in the notebook variable `API_KEY` for this session.
- The notebook must remain running for the endpoint to remain available.
- This is an experimental endpoint. Do not use it for sensitive data. Verify client protocol compatibility and keep the API key private.